# 📘 Notebook B — Supervised Fine-Tuning (SFT)
## Address Normalization with QLoRA + Unsloth

**Goal**: Fine-tune Llama-3.1-8B to parse messy Indonesian addresses into structured JSON.

**Pipeline**: `training_dataset_v2.csv` → Chat-formatted dataset → QLoRA SFT → LoRA adapter weights

> ⚠️ **Run this notebook on Google Colab (T4 GPU)**. Free tier is sufficient with 4-bit quantization.

In [2]:
%%capture
# ============================================================================
# 1. INSTALL DEPENDENCIES (Run once per Colab session)
# ============================================================================
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets wandb

In [3]:
# ============================================================================
# 2. CONFIGURATION — All hyperparameters in one place
# ============================================================================
import os

# ── Paths ──
# If running on Colab, upload training_dataset_v2.csv to /content/ first
# Or mount Google Drive and point to the file there
TRAINING_CSV = "training_dataset_v2.csv"  # Update this path as needed
OUTPUT_DIR = "address-parser-llama3"       # Where adapter weights are saved
HF_REPO_ID = None  # Set to "your-username/address-parser" to push to HuggingFace

# ── Model ──
BASE_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # Pre-quantized
MAX_SEQ_LENGTH = 1024   # Our JSON outputs are ~300 tokens, 1024 is plenty
DTYPE = None            # Auto-detect (float16 on T4, bfloat16 on A100)

# ── LoRA ──
LORA_R = 16             # Rank — 16 is a good balance for structured tasks
LORA_ALPHA = 16          # Scaling factor (alpha/r = 1.0)
LORA_DROPOUT = 0         # Unsloth optimized — keep at 0

# ── Training ──
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4           # Effective batch = 2 * 4 = 8
LEARNING_RATE = 2e-4
WARMUP_STEPS = 50
LOGGING_STEPS = 25
SAVE_STEPS = 200
EVAL_SPLIT = 0.05        # 5% held out for validation
SEED = 42

print("✅ Config loaded")

✅ Config loaded


In [4]:
# ============================================================================
# 3. LOAD BASE MODEL + ATTACH LoRA ADAPTERS
# ============================================================================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Memory optimization
    random_state=SEED,
)

print(f"✅ Model loaded: {BASE_MODEL}")
print(f"   Trainable params: {model.print_trainable_parameters()}")

NotImplementedError: Unsloth currently only works on NVIDIA, AMD and Intel GPUs.

In [ ]:
# ============================================================================
# 4. PREPARE DATASET — Convert CSV to Chat Format
# ============================================================================
import pandas as pd
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Apply Llama-3.1 chat template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

# ── System prompt: teaches the model its role ──
SYSTEM_PROMPT = """Kamu adalah mesin parsing alamat Indonesia. Tugasmu adalah mengekstrak komponen alamat dari teks informal dan mengembalikan hasilnya dalam format JSON yang valid.

Aturan:
1. Ekstrak setiap komponen yang ada: nama_jalan, nama_kompleks_atau_gedung, blok_kavling, nomor, rt, rw, kelurahan, kecamatan, kota_kabupaten, provinsi, kodepos, detail_unit
2. Jika suatu komponen TIDAK disebutkan dalam teks, isi dengan null. JANGAN menebak atau mengarang.
3. Normalisasi kapitalisasi ke Title Case (contoh: "jl. sudirman" → "Jl. Sudirman")
4. Kembalikan HANYA objek JSON, tanpa teks tambahan."""

# ── Load and convert CSV ──
df = pd.read_csv(TRAINING_CSV)
print(f"📊 Loaded {len(df)} training examples")

def row_to_conversation(row):
    """Convert one CSV row into a Llama-3.1 chat conversation."""
    return {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": str(row["input_text"])},
            {"role": "assistant", "content": str(row["output_json"])},
        ]
    }

# Build conversations list
records = [row_to_conversation(row) for _, row in df.iterrows()]
dataset = Dataset.from_list(records)

# Apply chat template formatting
def format_chat(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in convos
    ]
    return {"text": texts}

dataset = dataset.map(format_chat, batched=True)

# Train/eval split
split = dataset.train_test_split(test_size=EVAL_SPLIT, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"✅ Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
print(f"\n📝 Sample formatted prompt (first 500 chars):")
print(train_dataset[0]["text"][:500])

In [ ]:
# ============================================================================
# 5. TRAINING — QLoRA SFT with Unsloth-optimized SFTTrainer
# ============================================================================
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,  # Disable packing for structured output tasks
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        warmup_steps=WARMUP_STEPS,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        eval_strategy="steps",
        eval_steps=SAVE_STEPS,
        save_total_limit=3,
        fp16=not __import__('torch').cuda.is_bf16_supported(),
        bf16=__import__('torch').cuda.is_bf16_supported(),
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        report_to="none",  # Set to "wandb" if you want W&B logging
    ),
)

print("🚀 Starting training...")
stats = trainer.train()
print(f"\n✅ Training complete!")
print(f"   Total steps: {stats.global_step}")
print(f"   Final loss:  {stats.training_loss:.4f}")

In [ ]:
# ============================================================================
# 6. SAVE ADAPTER WEIGHTS
# ============================================================================

# Save LoRA adapter (small, ~50-100MB)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ LoRA adapter saved to {OUTPUT_DIR}/")

# Optional: Push to HuggingFace Hub
if HF_REPO_ID:
    model.push_to_hub(HF_REPO_ID, token=os.environ.get("HF_TOKEN"))
    tokenizer.push_to_hub(HF_REPO_ID, token=os.environ.get("HF_TOKEN"))
    print(f"✅ Pushed to https://huggingface.co/{HF_REPO_ID}")

# Optional: Export to GGUF for local inference with Ollama
# Uncomment below if you want a GGUF file
# model.save_pretrained_gguf(OUTPUT_DIR + "-gguf", tokenizer, quantization_method="q4_k_m")
# print(f"✅ GGUF saved to {OUTPUT_DIR}-gguf/")

In [ ]:
# ============================================================================
# 7. QUICK INFERENCE TEST — Verify the model works before closing
# ============================================================================
FastLanguageModel.for_inference(model)

test_inputs = [
    "kirim ke jl. sudirman no 15 rt 3 rw 7 menteng jakpus",
    "Kalibata City Tower A lantai 12, Rawajati, Pancoran, Jakarta Selatan 12750",
    "Gg. Melati No.5, RT 02/RW 08, Cipete Selatan, Cilandak, Jaksel",
    "alamat saya di komp taman aries blok b3 no 22 kembangan jakbar",
]

for i, test in enumerate(test_inputs):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": test},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    output = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.1)
    response = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)

    print(f"\n{'='*60}")
    print(f"[Test {i+1}] INPUT: {test}")
    print(f"[Test {i+1}] OUTPUT:")
    print(response)

## 📥 Next Steps

### Download the adapter weights
```python
# On Colab, download the adapter folder
from google.colab import files
!zip -r address-parser-llama3.zip address-parser-llama3/
files.download("address-parser-llama3.zip")
```

### Use in Notebook C (Inference + RAG)
The adapter folder contains:
- `adapter_model.safetensors` — The trained LoRA weights
- `adapter_config.json` — LoRA configuration
- `tokenizer.json` + `tokenizer_config.json` — Tokenizer files

Load it in Notebook C with:
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="address-parser-llama3",  # Path to adapter folder
    max_seq_length=1024,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```